# Omni-Stack Training Pod — Quick-Start Notebook

This notebook walks you through fine-tuning a language model using **QLoRA** (via [Unsloth](https://github.com/unslothai/unsloth) + [TRL SFTTrainer](https://huggingface.co/docs/trl)) on the datasets that were pre-downloaded to `/workspace/datasets/`.

**Sections:**
1. Environment check
2. Choose your base model & dataset
3. Load model with 4-bit quantization (QLoRA)
4. Prepare dataset
5. Configure & run SFTTrainer
6. Save / merge adapter weights
7. (Optional) DPO fine-tuning
8. (Optional) GRPO fine-tuning

## 1. Environment Check

In [ ]:
import os, subprocess, torch

print('=== GPU Info ===')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'])

print(f'\nPyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU(s)          : {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} — '
              f'{torch.cuda.get_device_properties(i).total_memory // (1024**3)} GB')

print('\n=== Datasets on disk ===')
datasets_dir = '/workspace/datasets'
if os.path.isdir(datasets_dir):
    for d in sorted(os.listdir(datasets_dir)):
        print(f'  {d}')
else:
    print('  No datasets found — check /workspace/logs/dataset_downloads.log')

print('\n=== Model cache ===')
hf_cache = '/workspace/hf_cache'
if os.path.isdir(hf_cache):
    for d in sorted(os.listdir(hf_cache)):
        print(f'  {d}')
else:
    print('  No cached models found')

## 2. Configuration — choose your model & dataset

In [ ]:
import os

# ── Base model (must already be in /workspace/hf_cache/) ─────────────────────
BASE_MODEL = os.getenv(
    'TRAIN_BASE_MODEL',
    'huihui-ai/Huihui-Qwen3-Next-80B-A3B-Instruct-abliterated'
)
MODEL_NAME  = BASE_MODEL.split('/')[-1]
MODEL_PATH  = f'/workspace/hf_cache/{MODEL_NAME}'

# ── Dataset (folder under /workspace/datasets/) ───────────────────────────────
DATASET_NAME = os.getenv('TRAIN_DATASET', 'OpenHermes-2.5')
DATASET_PATH = f'/workspace/datasets/{DATASET_NAME}'

# ── LoRA hyper-parameters ─────────────────────────────────────────────────────
LORA_RANK       = int(os.getenv('LORA_RANK', '64'))
LORA_ALPHA      = int(os.getenv('LORA_ALPHA', '128'))
MAX_SEQ_LENGTH  = int(os.getenv('MAX_SEQ_LENGTH', '4096'))

# ── Training hyper-parameters ─────────────────────────────────────────────────
BATCH_SIZE      = int(os.getenv('TRAIN_BATCH_SIZE', '2'))
GRAD_ACCUM      = int(os.getenv('GRAD_ACCUM_STEPS', '8'))
LEARNING_RATE   = float(os.getenv('LEARNING_RATE', '2e-4'))
NUM_EPOCHS      = int(os.getenv('NUM_TRAIN_EPOCHS', '1'))

OUTPUT_DIR      = '/workspace/output/checkpoints'
FINAL_MODEL_DIR = '/workspace/output/final_model'

print('Configuration:')
print(f'  Model path    : {MODEL_PATH}')
print(f'  Dataset path  : {DATASET_PATH}')
print(f'  LoRA rank     : {LORA_RANK}  alpha: {LORA_ALPHA}')
print(f'  Batch size    : {BATCH_SIZE}  grad_accum: {GRAD_ACCUM}')
print(f'  Learning rate : {LEARNING_RATE}  epochs: {NUM_EPOCHS}')
print(f'  Max seq len   : {MAX_SEQ_LENGTH}')
print(f'  Output dir    : {OUTPUT_DIR}')

## 3. Load model with 4-bit QLoRA (Unsloth)

Unsloth loads the model in 4-bit using bitsandbytes and wraps it with LoRA adapters automatically.

In [ ]:
try:
    from unsloth import FastLanguageModel
    USE_UNSLOTH = True
    print('Unsloth available — using fast kernels.')
except ImportError:
    USE_UNSLOTH = False
    print('Unsloth not installed — falling back to standard transformers + bitsandbytes.')

import torch

if USE_UNSLOTH:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,          # auto-detect bf16 / fp16
        load_in_4bit=True,   # QLoRA
    )
    if LORA_RANK > 0:
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_RANK,
            lora_alpha=LORA_ALPHA,
            lora_dropout=0.05,
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                            'gate_proj', 'up_proj', 'down_proj'],
            bias='none',
            use_gradient_checkpointing='unsloth',
        )
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=bnb_config,
        device_map='auto',
    )
    if LORA_RANK > 0:
        lora_config = LoraConfig(
            r=LORA_RANK,
            lora_alpha=LORA_ALPHA,
            lora_dropout=0.05,
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
            bias='none',
            task_type='CAUSAL_LM',
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

print('Model loaded.')

## 4. Prepare dataset

Loads the dataset from disk and converts it to a chat-formatted prompt.

In [ ]:
from datasets import load_from_disk, load_dataset
import os

# Try loading from pre-downloaded local copy first.
if os.path.isdir(DATASET_PATH):
    try:
        dataset = load_from_disk(DATASET_PATH)
        print(f'Loaded from disk: {DATASET_PATH}')
    except Exception as e:
        print(f'load_from_disk failed ({e}), trying load_dataset with data_dir...')
        dataset = load_dataset('parquet', data_dir=DATASET_PATH, split='train')
else:
    # Fall back to streaming from HuggingFace Hub
    print(f'Dataset not on disk, streaming from HuggingFace: {DATASET_NAME}')
    dataset = load_dataset(DATASET_NAME, split='train', streaming=False)

# Grab the train split if the dataset is a DatasetDict
if hasattr(dataset, 'keys'):
    split_names = list(dataset.keys())
    print(f'Available splits: {split_names}')
    dataset = dataset[split_names[0]]

print(f'Dataset size: {len(dataset):,} examples')
print(f'Columns: {dataset.column_names}')
print('Sample:', dataset[0])

In [ ]:
# ── Formatting function ────────────────────────────────────────────────────────
# Adjust column names to match your dataset.
# Common column patterns:
#   instruction / input / output  (Alpaca)
#   conversations                 (ShareGPT)
#   prompt / chosen               (UltraFeedback / DPO)

def format_chat_sample(sample):
    """Convert a dataset row to a tokenizer chat string."""
    cols = set(sample.keys())

    if 'conversations' in cols:          # ShareGPT format
        messages = []
        for turn in sample['conversations']:
            role = 'user' if turn.get('from') in ('human', 'user') else 'assistant'
            messages.append({'role': role, 'content': turn.get('value', '')})
        return tokenizer.apply_chat_template(messages, tokenize=False,
                                             add_generation_prompt=False)

    if 'instruction' in cols:            # Alpaca format
        inp = sample.get('input', '') or ''
        prompt = sample['instruction']
        if inp:
            prompt = f'{prompt}\n\n{inp}'
        messages = [
            {'role': 'user',      'content': prompt},
            {'role': 'assistant', 'content': sample.get('output', '')},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False,
                                             add_generation_prompt=False)

    if 'prompt' in cols and 'chosen' in cols:  # DPO/UltraFeedback
        messages = [
            {'role': 'user',      'content': sample['prompt']},
            {'role': 'assistant', 'content': sample['chosen']},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False,
                                             add_generation_prompt=False)

    # Fallback: concatenate all string-valued columns
    return ' '.join(str(v) for v in sample.values() if isinstance(v, str))


# Preview one formatted sample
print(format_chat_sample(dataset[0])[:500])

## 5. SFT Training (TRL SFTTrainer)

Uses the chat-formatted dataset to run supervised fine-tuning.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
import os

use_wandb = bool(os.getenv('WANDB_API_KEY', '').strip())

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim='adamw_8bit',
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    max_seq_length=MAX_SEQ_LENGTH,
    logging_steps=25,
    save_strategy='epoch',
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to='wandb' if use_wandb else 'tensorboard',
    dataset_text_field=None,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    formatting_func=format_chat_sample,
    args=sft_config,
)

print('Starting training...')
trainer.train()
print('Training complete!')

## 6. Save adapter weights

Saves the LoRA adapter (fast, small) **and** optionally merges it into the base weights (slower, full precision).

In [ ]:
import os

# Always save the LoRA adapter
adapter_dir = '/workspace/output/lora_adapter'
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'LoRA adapter saved to {adapter_dir}')

# Merge adapter into base weights (requires enough VRAM for the full model)
MERGE_MODEL = os.getenv('MERGE_AFTER_TRAIN', 'false').lower() == 'true'

if MERGE_MODEL:
    print('Merging LoRA adapter into base model weights...')
    if USE_UNSLOTH:
        model.save_pretrained_merged(FINAL_MODEL_DIR, tokenizer,
                                     save_method='merged_16bit')
    else:
        merged_model = model.merge_and_unload()
        merged_model.save_pretrained(FINAL_MODEL_DIR)
        tokenizer.save_pretrained(FINAL_MODEL_DIR)
    print(f'Merged model saved to {FINAL_MODEL_DIR}')
else:
    print('Skipping merge. Set MERGE_AFTER_TRAIN=true to merge LoRA into base weights.')

## 7. (Optional) DPO Fine-Tuning

Use this section instead of the SFT section above when you have a **preference dataset** with `prompt`, `chosen`, and `rejected` columns (e.g., UltraFeedback).

In [ ]:
# Uncomment to run DPO instead of SFT

# from trl import DPOTrainer, DPOConfig
#
# dpo_config = DPOConfig(
#     output_dir=OUTPUT_DIR,
#     num_train_epochs=NUM_EPOCHS,
#     per_device_train_batch_size=BATCH_SIZE,
#     gradient_accumulation_steps=GRAD_ACCUM,
#     learning_rate=LEARNING_RATE,
#     max_length=MAX_SEQ_LENGTH,
#     logging_steps=25,
#     save_strategy='epoch',
#     bf16=torch.cuda.is_bf16_supported(),
#     report_to='wandb' if use_wandb else 'tensorboard',
# )
#
# dpo_trainer = DPOTrainer(
#     model=model,
#     ref_model=None,  # None → implicit reference from the base model
#     tokenizer=tokenizer,
#     train_dataset=dataset,
#     args=dpo_config,
# )
#
# dpo_trainer.train()

## 8. (Optional) GRPO Fine-Tuning (Reasoning / RL)

Use GRPO when you have a **verifiable reward signal** (e.g., math problems, code execution).

In [ ]:
# Uncomment to run GRPO reinforcement learning from a reward function

# from trl import GRPOTrainer, GRPOConfig
#
# def reward_fn(completions, **kwargs):
#     """Return a list of scalar rewards, one per completion."""
#     rewards = []
#     for comp in completions:
#         text = comp[0]['content'] if isinstance(comp, list) else comp
#         # ---- insert your reward logic here ----
#         score = 1.0 if 'correct' in text.lower() else 0.0
#         rewards.append(score)
#     return rewards
#
# grpo_config = GRPOConfig(
#     output_dir=OUTPUT_DIR,
#     num_train_epochs=NUM_EPOCHS,
#     per_device_train_batch_size=BATCH_SIZE,
#     gradient_accumulation_steps=GRAD_ACCUM,
#     learning_rate=LEARNING_RATE,
#     logging_steps=25,
#     bf16=torch.cuda.is_bf16_supported(),
#     report_to='wandb' if use_wandb else 'tensorboard',
# )
#
# grpo_trainer = GRPOTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     reward_funcs=reward_fn,
#     train_dataset=dataset,
#     args=grpo_config,
# )
#
# grpo_trainer.train()